# Model 2 — Complex CNN

In [ ]:
# %pip install --no-cache-dir --retries 10 --timeout 120 tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 1.4 MB/s  0:08:22m0:00:0100:13m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 774.4 kB/s  0:00:06 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 1.0 MB/s  0:00:02 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 971.8 kB/s  0:00:25m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14/14 [tensorflow]4 [tensorflow]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# !pip install scikit-learn

In [1]:
import os
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras import layers, models

DATASET = "raw"

# In VS Code the notebook kernel may start in either the project root or src/.
ROOT = Path.cwd().resolve()
if not (ROOT / "data" / DATASET).is_dir() and (ROOT.parent / "data" / DATASET).is_dir():
    ROOT = ROOT.parent

data_dir = ROOT / "data" / DATASET
if not data_dir.is_dir():
    raise FileNotFoundError(f"Cannot find dataset folder: {data_dir}")

print(f"Project root: {ROOT}")
print(f"Dataset: {data_dir}")

I0000 00:00:1791472108.166373   64527 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791472109.281716   64527 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Project root: /home/pc/AAA/project/deepnn/G8_AI66A_DNN
Dataset: /home/pc/AAA/project/deepnn/G8_AI66A_DNN/data/raw


## 1. Build the network

In [2]:
def build_model(number_of_classes):
    model = models.Sequential(name="complex_cnn")
    model.add(layers.Input(shape=(*IMAGE_SIZE, 3)))

    model.add(layers.RandomFlip("horizontal"))
    model.add(layers.RandomRotation(0.02))
    model.add(layers.RandomZoom(0.05))
    model.add(layers.Rescaling(1.0 / 255))

    # CNN block 1
    model.add(layers.Conv2D(32, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))

    # CNN block 2
    model.add(layers.Conv2D(64, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))

    # CNN block 3
    model.add(layers.Conv2D(128, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))

    # CNN block 4
    model.add(layers.Conv2D(256, (3, 3), padding="same", use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.Activation("relu"))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.3))

    # Classifier
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(128, activation="relu"))
    model.add(layers.Dropout(0.4))
    model.add(layers.Dense(number_of_classes, activation="softmax"))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

## 2. Load train, validation and test data

Set the input image size, batch size and number of training epochs here. The directory names become class labels; the loader resizes images and creates one-hot labels to match `categorical_crossentropy`.

In [3]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
EPOCHS = 15

split_dirs = {split: data_dir / split for split in ("train", "val", "test")}
missing_dirs = [str(path) for path in split_dirs.values() if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(f"Missing dataset split folders: {missing_dirs}")

train_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["train"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["val"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    split_dirs["test"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False,
)

class_names = train_ds.class_names
if val_ds.class_names != class_names or test_ds.class_names != class_names:
    raise ValueError("Train, validation and test class folders must match.")

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

print("Classes:", class_names)

Found 76515 files belonging to 4 classes.
Found 21861 files belonging to 4 classes.
Found 10933 files belonging to 4 classes.
Classes: ['CNV', 'DME', 'DRUSEN', 'NORMAL']


## 3. Create the model

In [4]:
model = build_model(number_of_classes=len(class_names))
images, labels = next(iter(train_ds))
print("Input batch:", images.shape)
print("Label batch:", labels.shape)
print("Model output:", model(images, training=False).shape)
print(f"Trainable parameters: {model.count_params():,}")

Input batch: (64, 224, 224, 3)
Label batch: (64, 4)
Model output: (64, 4)
Trainable parameters: 423,268


## 4. Train

In [5]:
model_path = ROOT / "saved_models" / f"complex_cnn_{DATASET}.keras"
model_path.parent.mkdir(parents=True, exist_ok=True)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.ModelCheckpoint(
            str(model_path),
            monitor="val_loss",
            save_best_only=True,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=4,
            restore_best_weights=True,
        ),
    ],
)

print("Best checkpoint:", model_path)

Epoch 1/15


/home/pc/miniconda3/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


  48/1196 ━━━━━━━━━━━━━━━━━━━━ 15:19 801ms/step - accuracy: 0.6016 - loss: 1.0628

KeyboardInterrupt: 

## 4. Evaluate on the held-out test set

In [19]:
best_model = tf.keras.models.load_model(model_path)
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)
y_true = np.concatenate([np.argmax(labels.numpy(), axis=1) for _, labels in test_ds])
y_pred = np.argmax(best_model.predict(test_ds, verbose=0), axis=1)

print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_accuracy:.4f}")
print("\nPrecision / Recall / F1-score:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0,
))
print("Confusion matrix (rows=true, columns=predicted):")
print(confusion_matrix(y_true, y_pred))

Test loss: 1.3686 | Test accuracy: 0.3400

Precision / Recall / F1-score:
              precision    recall  f1-score   support

         CNV       0.34      1.00      0.51        34
         DME       0.00      0.00      0.00        11
      DRUSEN       0.00      0.00      0.00         8
      NORMAL       0.00      0.00      0.00        47

    accuracy                           0.34       100
   macro avg       0.09      0.25      0.13       100
weighted avg       0.12      0.34      0.17       100

Confusion matrix (rows=true, columns=predicted):
[[34  0  0  0]
 [11  0  0  0]
 [ 8  0  0  0]
 [47  0  0  0]]
